# Predictive Modeling for Customer Churn

End-to-end supervised machine learning project for churn prediction, built for GitHub submission and the Thiranex internship task.

## Objective

Build a professional churn prediction workflow that cleans the data, performs exploratory analysis, trains multiple supervised learning models, compares their performance, and saves all results and visualizations.

## Dataset

The dataset is `customer_churn_data.csv`. It contains customer profile, service usage, billing, and contract variables, with `Churn` as the binary target. The identifier column `customerID` is removed during preprocessing.

In [ ]:
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

from predictive_model import (
    BASE_DIR,
    build_results_summary,
    clean_dataset,
    inspect_data,
    infer_target_column,
    load_dataset,
    plot_boxplots,
    plot_correlation_heatmap,
    plot_feature_distribution,
    plot_target_distribution,
    split_features_target,
    train_and_evaluate_models,
)

data_path = BASE_DIR / 'customer_churn_data.csv'
raw_df = load_dataset(data_path)
target_column = infer_target_column(raw_df)

inspect_data(raw_df, title='Raw Dataset Overview')
print(f'\nTarget column: {target_column}')

## Data Cleaning

This step removes duplicate rows, converts numeric-like fields safely, and drops identifier columns that do not help prediction.

In [ ]:
cleaned_df, dropped_identifier_columns = clean_dataset(raw_df, target_column)
print(f'Dropped identifier columns: {dropped_identifier_columns if dropped_identifier_columns else None}')
inspect_data(cleaned_df, title='Cleaned Dataset Overview')

features, target_encoded, label_encoder = split_features_target(cleaned_df, target_column)
print(f'\nFeature matrix shape: {features.shape}')
print(f'Target classes: {list(label_encoder.classes_)}')
print(f'Encoded target balance: {dict(zip(label_encoder.classes_, np.bincount(target_encoded)))}')

## EDA

The notebook and script save all EDA figures into `outputs/` for submission readiness and reuse.

In [ ]:
numeric_features = features.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = features.select_dtypes(exclude=[np.number]).columns.tolist()
output_dir = BASE_DIR / 'outputs'

plot_target_distribution(cleaned_df, target_column, output_dir / 'target_distribution.png')
plot_correlation_heatmap(cleaned_df, target_column, output_dir / 'correlation_heatmap.png')
plot_feature_distribution(cleaned_df, target_column, numeric_features, categorical_features, output_dir / 'feature_distribution.png')
plot_boxplots(cleaned_df, target_column, numeric_features, output_dir / 'boxplots.png')

print('EDA plots saved to outputs/.')

## Feature Engineering

Features are split automatically into numeric and categorical subsets, then prepared with imputation, scaling, and one-hot encoding inside a reusable preprocessing pipeline.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    features,
    target_encoded,
    test_size=0.2,
    random_state=42,
    stratify=target_encoded,
)

print(f'Train shape: {x_train.shape}')
print(f'Test shape: {x_test.shape}')

## Model Training and Evaluation

The project trains Logistic Regression, Decision Tree Classifier, and Random Forest Classifier, then compares them using accuracy, precision, recall, F1 score, and ROC AUC.

In [ ]:
comparison_df, trained_models, evaluation_details = train_and_evaluate_models(
    x_train,
    x_test,
    y_train,
    y_test,
    output_dir,
)

comparison_df.round(3)

## Results

The best model is selected automatically from the comparison table. Its confusion matrix, ROC curve, feature importance plot, and summary file are all saved in `outputs/`.

In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
print(f'Best model: {best_model_name}')
print('\nClassification report:')
print(evaluation_details[best_model_name]['classification_report'])
print('\nWhy it performed best:')
print(build_results_summary(comparison_df))

## Conclusion

This project delivers a complete churn prediction workflow with cleaned data, reusable preprocessing, multiple trained models, and professional result visualizations.

## Future Improvements

Potential next steps include hyperparameter tuning, cross-validation, threshold optimization, and additional explainability techniques such as SHAP values.